In [ ]:
# Download data

In [ ]:
import os
import polars as pl
import numpy as np
import plotly.express as px
from typing import List, Dict, Callable, Any
from tqdm import tqdm
#import datasets

In [ ]:
# import polars as pl
# import numpy as np

# PARQUET_PATH = "/cs/labs/oabend/tomer.shahaf/hf_cache_root/sharelm_full.parquet"
# N_ROWS = 100_000

# # Lazy load
# lf = pl.scan_parquet(PARQUET_PATH)

# # Need total rows
# total_rows = lf.collect().shape[0]  # can cache this somewhere

# # Random sample
# random_indices = np.random.choice(total_rows, size=N_ROWS, replace=False)

# df_sampled = lf.with_row_count("row_nr").filter(
#     pl.col("row_nr").is_in(random_indices)
# ).collect()


In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
# Get the CACHE_ROOT path from the environment variable you just set in Linux
CACHE_ROOT = os.environ.get("HF_HOME")

if CACHE_ROOT is None:
    # This should not happen, but a safe fallback in case the variable was lost
    CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
    os.environ["HF_HOME"] = CACHE_ROOT

In [ ]:
research_df_parquet_path = os.path.join(CACHE_ROOT, "df_sampled_100k.pqt")
research_df_tmp_parquet_path = os.path.join(CACHE_ROOT, "df_sampled_100k_tmp.pqt")
sharelm_full_parquet_path = os.path.join(CACHE_ROOT, "sharelm_full.parquet")
research_df_tmp_with_cosines_parquet_path = os.path.join(CACHE_ROOT, "df_sampled_100k_tmp_with_cosines.pqt")
research_high_semantic_conversations_parquet_path = os.path.join(CACHE_ROOT, "research_high_semantic_conversations.pqt")
dataset_cache_dir = os.path.join(CACHE_ROOT, "datasets")

In [ ]:
# load with datasets
ds = datasets.load_dataset(
    "shachardon/ShareLM", 
    split="train", 
    cache_dir=dataset_cache_dir
)

In [ ]:
# load research df
research_df = pl.read_parquet(research_df_parquet_path)
research_df.shape

In [ ]:
# lazt load full df
lazy_full_df = pl.scan_parquet(sharelm_full_parquet_path)

num_cols = len(lazy_full_df.columns)
num_rows = lazy_full_df.select(pl.len()).collect().item()

print(f"Shape: ({num_rows}, {num_cols})")

In [ ]:
import polars as pl
import plotly.express as px

def plot_model_distribution(df: pl.DataFrame | pl.LazyFrame, n: int = 10):
    # 1. Normalize to LazyFrame (works for both eager & lazy)
    lf = df.lazy()

    # 2. Pure lazy pipeline
    counts_lf = (
        lf.group_by("model_name")
          .len()
          .rename({"len": "count"})
          .sort("count", descending=True)
    )

    # 3. Collect only once (boundary: plotting)
    counts = counts_lf.collect()

    # 4. Post-processing (small data now, safe in RAM)
    top_n = counts.head(n)
    other_val = counts.slice(n).select(pl.col("count").sum()).item()

    if other_val > 0:
        other_row = pl.DataFrame(
            {"model_name": ["Other"], "count": [other_val]},
            schema=top_n.schema
        )
        top_n = top_n.vstack(other_row)

    # 5. Plot
    fig = px.pie(
        top_n.to_pandas(),
        values="count",
        names="model_name",
        hole=0.3,
    )
    fig.update_layout(title=f"Top {n} Models")
    fig.show()


In [ ]:
plot_model_distribution(research_df)

In [ ]:
plot_model_distribution(lazy_full_df)

In [ ]:
import polars as pl
import plotly.express as px


def plot_top_users(
    df: pl.DataFrame | pl.LazyFrame,
    column_name: str = "user_id",
    n: int = 10,
):
    # 1. Normalize to LazyFrame
    lf = df.lazy()

    # 2. Lazy aggregation (streaming-friendly)
    counts_lf = (
        lf.group_by(column_name)
          .len()
          .rename({"len": "count"})
          .sort("count", descending=True)
    )

    # 3. Collect once (boundary)
    counts = counts_lf.collect(streaming=True)

    # 4. Top-N + Other (small data now)
    top_n = counts.head(n)

    other_count = (
        counts
        .slice(n)
        .select(pl.col("count").sum())
        .item()
    )

    if other_count > 0:
        other_row = pl.DataFrame(
            {column_name: ["Other"], "count": [other_count]},
            schema=top_n.schema,
        )
        plot_df = top_n.vstack(other_row)
    else:
        plot_df = top_n

    # 5. Plot
    fig = px.pie(
        plot_df.to_pandas(),
        values="count",
        names=column_name,
        hole=0.3,
    )

    fig.update_layout(
        title={
            "text": f"Top {n} {column_name}s",
            "font": {"size": 14},
        }
    )

    fig.show()

In [ ]:
plot_top_users(research_df)

In [ ]:
plot_top_users(lazy_full_df)

In [ ]:
import polars as pl
import plotly.express as px


def plot_events_by_year_month(
    df: pl.DataFrame | pl.LazyFrame,
    column_name: str = "timestamp",
    output_column: str = "YearMonth",
    date_format: str = "%Y-%m-%d %H:%M:%S.%f",
):
    # 1. Normalize to LazyFrame
    lf = df.lazy()

    # 2. Lazy transformation + aggregation
    counts_lf = (
        lf.with_columns(
            pl.col(column_name)
            .str.replace("Z", "")
            .str.strptime(
                pl.Datetime,
                format=date_format,
                strict=False,
            )
            .dt.strftime("%Y-%m")
            .fill_null("Unknown")
            .alias(output_column)
        )
        .group_by(output_column)
        .len()
        .rename({"len": "Count"})
        .filter(pl.col(output_column) != "Unknown")
        .sort(output_column)
    )

    # 3. Collect once (streaming-safe)
    result = counts_lf.collect(streaming=True)

    # 4. Plot
    fig = px.bar(
        result.to_pandas(),
        x=output_column,
        y="Count",
        title="Count of Events by Year and Month",
        labels={output_column: "Month", "Count": "Event Count"},
    )

    fig.show()


In [ ]:
plot_events_by_year_month(research_df)

In [ ]:
plot_events_by_year_month(lazy_full_df)

In [ ]:
import polars as pl
import plotly.express as px


def plot_conversation_length_distribution(
    df: pl.DataFrame | pl.LazyFrame,
    column_name: str = "conversation",
    x_max: int = 50,
):
    # 1. Normalize to LazyFrame
    lf = df.lazy()

    # 2. Lazy length computation + aggregation
    counts_lf = (
        lf.with_columns(
            pl.col(column_name)
            .list.len()
            .alias("conversation_len")
        )
        .group_by("conversation_len")
        .len()
        .rename({"len": "count"})
        .sort("conversation_len")
    )

    # 3. Collect once (streaming-safe)
    dist_df = counts_lf.collect(streaming=True)

    # Optional: clip to X_MAX for plotting
    dist_df = dist_df.filter(pl.col("conversation_len") <= x_max)

    # 4. Plot
    fig = px.bar(
        dist_df.to_pandas(),
        x="conversation_len",
        y="count",
        title="Distribution of Conversation Lengths",
        labels={
            "conversation_len": "Conversation Length (turns)",
            "count": "Frequency (Count)",
        },
    )

    fig.update_traces(opacity=0.8)

    fig.update_xaxes(
        range=[0, x_max],
        tickfont={"size": 8},
        title_font={"size": 12},
    )

    fig.update_layout(bargap=0.1)

    fig.show()


In [ ]:
plot_conversation_length_distribution(research_df)

In [ ]:
plot_conversation_length_distribution(lazy_full_df)

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def apply_with_progress(
    df: pl.DataFrame,
    column: str,
    func: Callable,
    return_dtype: Any,
    new_column: str = None,
    desc: str = "Processing"
) -> pl.DataFrame:
    
    if new_column is None:
        new_column = column
    
    total_rows = len(df)
    
    with tqdm(total=total_rows, desc=desc) as pbar:
        def func_with_progress(value):
            result = func(value)
            pbar.update(1)
            return result
        
        return df.with_columns(
            pl.col(column).map_elements(
                func_with_progress,
                return_dtype=return_dtype
            ).alias(new_column)
        )

In [ ]:
def extract_user_prompts(conversation):
    return [
        turn["content"] 
        for turn in conversation 
        if turn["role"] == "user"
    ]

In [ ]:
total_rows = num_rows
chunk_size = 100_000

for chunk_idx, offset in tqdm(enumerate(range(0, total_rows, chunk_size)), desc="Processing chunks"):
    chunk = lazy_full_df.slice(offset, chunk_size).collect()
    chunk_path = os.path.join(CACHE_ROOT, f"chunks/chunk_{chunk_idx}.pqt")
    chunk.write_parquet(chunk_path)

In [ ]:
!ls -lh ../hf_cache_root/processed_chunks

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 2
MAX_THRESHOLD_CONVERSATION_LEN = 10

for chunk_idx, offset in tqdm(enumerate(range(0, total_rows, chunk_size)), desc="Processing chunks"):
    chunk_path = os.path.join(CACHE_ROOT, f"chunks/chunk_{chunk_idx}.pqt")
    chunk_df = pl.read_parquet(chunk_path)

    chunk_df = (
        chunk_df.with_columns(
            pl.col("conversation")
            .map_elements(extract_user_prompts, return_dtype=pl.List(pl.String))
            .alias("user_prompts")
        )
    )

    chunk_df = (
        chunk_df.with_columns(
            pl.col("user_prompts").list.len()
            .alias("user_prompts_count")
        )
    )

    chunk_df = chunk_df.filter(~pl.col("user_prompts").list.contains(None))
    chunk_df = chunk_df.filter(pl.col("user_prompts_count") >= MIN_THRESHOLD_CONVERSATION_LEN)
    chunk_df = chunk_df.filter(pl.col("user_prompts_count") <= MAX_THRESHOLD_CONVERSATION_LEN)
    print(chunk_df.shape)
    
    processed_chunk_path = os.path.join(CACHE_ROOT, f"processed_chunks/processed_chunk_{chunk_idx}.pqt")
    chunk_df.write_parquet(processed_chunk_path)

In [ ]:
research_df = apply_with_progress(
    df=research_df,
    column="conversation",
    func=extract_user_prompts,
    return_dtype=pl.List(pl.String),
    new_column="user_prompts",
    desc="Extracting User Prompts"
)

In [ ]:
lazy_full_df = (
    lazy_full_df.with_columns(
        pl.col("conversation")
        .map_elements(extract_user_prompts, return_dtype=pl.List(pl.String))
        .alias("user_prompts")
    )
)

In [ ]:
research_df = apply_with_progress(
    df=research_df,
    column="user_prompts",
    func=lambda prompts_list: len(prompts_list),
    return_dtype=pl.Int64,
    new_column="user_prompts_count",
    desc="Counting user prompts"
)

In [ ]:
lazy_full_df = (
    lazy_full_df.with_columns(
        pl.col("user_prompts").list.len()
            .alias("user_prompts_count")
    )
)

In [ ]:
BATCH_SIZE = 10_000
lazy_full_df_partial = lazy_full_df.slice(0, BATCH_SIZE).collect(streaming=True)

In [ ]:
lazy_full_df_partial.sample()["user_prompts", "user_prompts_count"]

In [ ]:
# counts = lazy_full_df.select("user_turn_count").collect(streaming=True)
# px.histogram(counts)

In [ ]:
# sink_path = os.path.join(CACHE_ROOT, "user_turns.parquet")
# out_lf.sink_parquet(sink_path)

In [ ]:
research_df = research_df.filter(pl.col("user_prompts").list.len() > 0)
research_df = research_df.filter(~pl.col("user_prompts").list.contains(None))
research_df = research_df.filter(pl.col("user_prompts_count") > 1)
research_df.shape

In [ ]:
# lazy_full_df = lazy_full_df.filter(pl.col("user_prompts").list.len() > 0)
# lazy_full_df = lazy_full_df.filter(~pl.col("user_prompts").list.contains(None))
# lazy_full_df = lazy_full_df.filter(pl.col("user_prompts_count") > 1)

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 2
MAX_THRESHOLD_CONVERSATION_LEN = 10

lazy_full_df = (
    lazy_full_df.with_columns(
        pl.col("user_prompts_count")
        .map_elements(lambda user_prompts_count: (user_prompts_count >= MIN_THRESHOLD_CONVERSATION_LEN) and (user_prompts_count <= MAX_THRESHOLD_CONVERSATION_LEN),
                      return_dtype=pl.Boolean)
        .alias("is_long_conversation")
    )
)

In [ ]:
lazy_full_df = (
    lazy_full_df.with_columns(
        pl.col("user_prompts")
        .map_elements(lambda prompts: all(is_valid_prompt(prompt) for prompt in prompts),
        return_dtype=pl.Boolean)
        .alias("is_valid_prompts")
    )
)

In [ ]:
BATCH_SIZE = 1_000
lazy_full_df_partial = lazy_full_df.slice(0, BATCH_SIZE).collect(streaming=True)

In [ ]:
lazy_full_df_partial.sample()["user_prompts", "user_prompts_count", "is_long_conversation", "is_valid_prompts"]

In [ ]:
filtered_research_df = lazy_full_df_partial.filter(
    (pl.col("is_long_conversation") == True) & (pl.col("is_valid_prompts") == True)
)

In [ ]:
filtered_research_df.shape

In [ ]:
lazy_full_df = lazy_full_df.filter(
    (pl.col("is_long_conversation") == True) & (pl.col("is_valid_prompts") == True)
)

In [ ]:
sink_path = os.path.join(CACHE_ROOT, "long_conversations.pqt")
lazy_full_df.sink_parquet(sink_path)

In [ ]:
import string

def is_valid_prompt(prompt: str) -> bool:
    allowed_chars = set(string.printable)
    if not prompt.strip():
        return False
    is_printable_prompt = all(ch in allowed_chars for ch in prompt)
    return is_printable_prompt

In [ ]:
research_df = filter_with_progress(
    df=research_df,
    condition=pl.col("user_prompts").map_elements(
        lambda prompts: all(is_valid_prompt(prompt) for prompt in prompts),
        return_dtype=pl.Boolean
    ),
    desc="Filtering invalid prompts"
)

In [ ]:
# user prompts

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 2
MAX_THRESHOLD_CONVERSATION_LEN = 10


In [ ]:
research_df = research_df.filter((pl.col("user_prompts_count") > MIN_THRESHOLD_CONVERSATION_LEN) & 
                                 (pl.col("user_prompts_count") < MAX_THRESHOLD_CONVERSATION_LEN))
research_df.shape

In [ ]:
def extract_model_answers(conversation):
    return [
        turn["content"] 
        for turn in conversation 
        if turn["role"] == "assistant"
    ]

In [ ]:
research_df = apply_with_progress(
    df=research_df,
    column="conversation",
    func=extract_model_answers,
    return_dtype=pl.List(pl.String),
    new_column="model_answers",
    desc="Extracting model Answers"
)

In [ ]:
# extract vectors

In [ ]:
research_df_with_cosines = pl.read_parquet(research_df_tmp_with_cosines_parquet_path)
research_df_with_cosines.shape

In [ ]:
research_df_with_cosines.columns

In [ ]:
research_df_with_cosines.sample()["user_prompts_similarity_to_first"]

In [ ]:
# 1. Sample in Polars -> Convert to Pandas immediately
df_pandas = (
    research_df_with_cosines
    .sample(n=50, seed=42)
    .with_row_index(name="row_id")
    .to_pandas()  # <--- Move to Pandas here
)

# 2. Explode
df_pandas = df_pandas.explode("user_prompts_similarity_to_first")

# 3. Generate Rank (using GroupBy in Pandas)
# We group by row_id and create a counter 1..N for each group
df_pandas["Rank"] = df_pandas.groupby("row_id").cumcount() + 1

# 4. Rename and Type
df_pandas = df_pandas.rename(columns={"user_prompts_similarity_to_first": "cosine_value"})
df_pandas["row_id"] = df_pandas["row_id"].astype(str)

fig = px.line(
    df_pandas,
    x="Rank",
    y="cosine_value",
    color="row_id",       # Now distinct colors because it's a string
    line_group="row_id",
    markers=True,
    title="user_prompts_similarity_to_first",
    hover_data={"row_id": True, "Rank": True, "cosine_value": ':.3f'}
)

fig.update_traces(mode="lines+markers")
fig.update_layout(
    xaxis_title="User prompt index",
    yaxis_title="Cosine similarity",
    legend_title="Sample ID",
    hovermode="closest"
)

fig.show()

In [ ]:
# 1. Sample in Polars -> Convert to Pandas immediately
df_pandas = (
    research_df_with_cosines
    .sample(n=50, seed=42)
    .with_row_index(name="row_id")
    .to_pandas()  # <--- Move to Pandas here
)

# 2. Explode
df_pandas = df_pandas.explode("user_prompts_sequential_similarity")

# 3. Generate Rank (using GroupBy in Pandas)
# We group by row_id and create a counter 1..N for each group
df_pandas["Rank"] = df_pandas.groupby("row_id").cumcount() + 1

# 4. Rename and Type
df_pandas = df_pandas.rename(columns={"user_prompts_sequential_similarity": "cosine_value"})
df_pandas["row_id"] = df_pandas["row_id"].astype(str)

fig = px.line(
    df_pandas,
    x="Rank",
    y="cosine_value",
    color="row_id",       # Now distinct colors because it's a string
    line_group="row_id",
    markers=True,
    title="user_prompts_sequential_similarity",
    hover_data={"row_id": True, "Rank": True, "cosine_value": ':.3f'}
)

fig.update_traces(mode="lines+markers")
fig.update_layout(
    xaxis_title="User prompt index",
    yaxis_title="Cosine similarity",
    legend_title="Sample ID",
    hovermode="closest"
)

fig.show()

In [ ]:
row_id = 18
print(df_pandas.iloc[row_id]["cosine_value"])
df_pandas.iloc[row_id]["user_prompts"]

In [ ]:
# Calculate Median and Mean instantly (handles empty lists automatically)
research_df_with_cosines = research_df_with_cosines.with_columns([
    pl.col("user_prompts_similarity_to_first").list.median().alias("user_prompts_similarity_to_first_median"),
    pl.col("user_prompts_similarity_to_first").list.mean().alias("user_prompts_similarity_to_first_mean"),
    pl.col("user_prompts_sequential_similarity").list.median().alias("user_prompts_sequential_similarity_median"),
    pl.col("user_prompts_sequential_similarity").list.mean().alias("user_prompts_sequential_similarity_mean")

])

In [ ]:
# 1. Extract column and convert to Pandas for Plotly
# We use .select() first to ensure we only convert the necessary data to Pandas RAM
df_plot = research_df_with_cosines.select(
    pl.col("user_prompts_similarity_to_first_mean").alias("cosine_mean")
).to_pandas()

# Bin settings (exact bin edges)
bin_settings = dict(
    start=-1,
    end=1,
    size=0.01
)

# Create histogram
fig = px.histogram(
    df_plot,
    x='cosine_mean',
    nbins=10,  # ignored because xbins overrides it
    histnorm='percent',
    title='Distribution of Mean Cosine Similarity of User Prompts From the First User Prompt',
    labels={'cosine_mean': 'Mean Cosine Similarity Score'},
    color_discrete_sequence=['#4C78A8']
)

# Apply explicit bin configuration
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black'))
)

# Layout styling
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Percent)',  # Note: Changed 'Count' to 'Percent' to match histnorm
    bargap=0.005,
    template='plotly_white'
)

fig.show()

In [ ]:
# 1. Extract column and convert to Pandas for Plotly
# We use .select() first to ensure we only convert the necessary data to Pandas RAM
df_plot = research_df_with_cosines.select(
    pl.col("user_prompts_sequential_similarity_mean").alias("cosine_mean")
).to_pandas()

# Bin settings (exact bin edges)
bin_settings = dict(
    start=-1,
    end=1,
    size=0.01
)

# Create histogram
fig = px.histogram(
    df_plot,
    x='cosine_mean',
    nbins=10,  # ignored because xbins overrides it
    histnorm='percent',
    title='Distribution of Mean Cosine Similarity of User Prompts Sequentially',
    labels={'cosine_mean': 'Mean Cosine Similarity Score'},
    color_discrete_sequence=['#4C78A8']
)

# Apply explicit bin configuration
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black'))
)

# Layout styling
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Percent)',  # Note: Changed 'Count' to 'Percent' to match histnorm
    bargap=0.005,
    template='plotly_white'
)

fig.show()

In [ ]:
SEMANTIC_CHANGE_COSINE_THRESHOLD = 0.85

def get_count_before_semantic_change(arr, threshold = SEMANTIC_CHANGE_COSINE_THRESHOLD):
    try:
        return next(i for i, x in enumerate(arr) if x < threshold)
    except StopIteration:
        
        return len(arr)

In [ ]:
a = [1, 0.86, 0.4, 0.9]
get_count_before_semantic_change(a)

In [ ]:
SEMANTIC_CHANGE_COSINE_THRESHOLD = 0.85

research_df_with_cosines = research_df_with_cosines.with_columns(
    pl.col("user_prompts_similarity_to_first")
    .list.eval(pl.element() < SEMANTIC_CHANGE_COSINE_THRESHOLD) 
    .map_elements(lambda x: x.arg_max() if x.any() else len(x), return_dtype=pl.Int64) 
    .alias("count_before_prompt_semantic_change")
)

In [ ]:
px.histogram(list(research_df_with_cosines["count_before_prompt_semantic_change"]))

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 2

research_df_with_cosines = research_df_with_cosines.filter(
    pl.col("count_before_prompt_semantic_change") >= MIN_THRESHOLD_CONVERSATION_LEN
)

print(f"number of high semantic conversations: {research_df_with_cosines.shape[0]}")
print(f"all with bigger then {SEMANTIC_CHANGE_COSINE_THRESHOLD} cosine and more at least the {MIN_THRESHOLD_CONVERSATION_LEN} user turns")

In [ ]:
# model answers

In [ ]:
# 1. Sample, Explode, and Rank
plot_df = (
    research_df_with_cosines
    .sample(n=50, seed=42)
    .with_row_index(name="row_id")
    # Explode first to avoid 'int_range' shape errors on variable length lists
    .explode("model_answers_similarity_to_first")
    .with_columns(
        # Generate Rank [1..N] for each group (row_id)
        Rank = pl.int_range(1, pl.len() + 1).over("row_id")
    )
    .rename({"model_answers_similarity_to_first": "cosine_value"})
)

# 2. Convert to Pandas for Plotly
df_pandas = plot_df.to_pandas()

# 3. VISUALIZATION FIX: Cast row_id to string for distinct colors (Discrete Legend)
df_pandas["row_id"] = df_pandas["row_id"].astype(str)

# 4. Plot
fig = px.line(
    df_pandas,
    x="Rank",
    y="cosine_value",
    color="row_id",
    line_group="row_id",
    markers=True,
    title="Cosine similarity scores from all to first model answers",
    hover_data={"row_id": True, "Rank": True, "cosine_value": ':.3f'}
)

fig.update_traces(mode="lines+markers")
fig.update_layout(
    xaxis_title="Model answer index",
    yaxis_title="Cosine similarity",
    legend_title="Sample ID",
    hovermode="closest"
)

fig.show()

In [ ]:
# Calculate Median and Mean instantly (handles empty lists automatically)
research_df_with_cosines = research_df_with_cosines.with_columns([
    pl.col("model_answers_similarity_to_first").list.median().alias("model_answers_similarity_to_first_median"),
    pl.col("model_answers_similarity_to_first").list.mean().alias("model_answers_similarity_to_first_mean"),
    pl.col("model_answers_sequential_similarity").list.median().alias("model_answers_sequential_similarity_median"),
    pl.col("model_answers_sequential_similarity").list.mean().alias("model_answers_sequential_similarity_mean")

])

In [ ]:
# 1. Extract column and convert to Pandas for Plotly
# We use .select() first to ensure we only convert the necessary data to Pandas RAM
df_plot = research_df_with_cosines.select(
    pl.col("model_answers_similarity_to_first_mean").alias("cosine_mean")
).to_pandas()

# Bin settings (exact bin edges)
bin_settings = dict(
    start=-1,
    end=1,
    size=0.01
)

# Create histogram
fig = px.histogram(
    df_plot,
    x='cosine_mean',
    nbins=10,  # ignored because xbins overrides it
    histnorm='percent',
    title='model_answers_similarity_to_first_mean',
    labels={'cosine_mean': 'Mean Cosine Similarity Score'},
    color_discrete_sequence=['#4C78A8']
)

# Apply explicit bin configuration
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black'))
)

# Layout styling
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Percent)',  # Note: Changed 'Count' to 'Percent' to match histnorm
    bargap=0.005,
    template='plotly_white'
)

fig.show()

In [ ]:
SEMANTIC_CHANGE_COSINE_THRESHOLD = 0.85

research_df_with_cosines = research_df_with_cosines.with_columns(
    pl.col("model_answers_similarity_to_first")
    .list.eval(pl.element() < SEMANTIC_CHANGE_COSINE_THRESHOLD) 
    .map_elements(lambda x: x.arg_max() if x.any() else len(x), return_dtype=pl.Int64) 
    .alias("count_before_model_semantic_change")
)

In [ ]:
px.histogram(list(research_df_with_cosines["count_before_model_semantic_change"]))

In [ ]:
research_df_with_cosines = research_df_with_cosines.filter(
    pl.col("count_before_model_semantic_change") >= MIN_THRESHOLD_CONVERSATION_LEN
)

print(f"number of high semantic conversations: {research_df_with_cosines.shape[0]}")
print(f"all with bigger then {SEMANTIC_CHANGE_COSINE_THRESHOLD} cosine and more at least the {MIN_THRESHOLD_CONVERSATION_LEN} user turns")

In [ ]:
research_df_with_cosines.shape

In [ ]:
research_df_with_cosines.write_parquet(research_high_semantic_conversations_parquet_path)